# Different Convergence Plot for the Ishigami Function

In [ ]:
import dill
import os
import chaospy as cp
import numpy as np
import math
import sys
import pathlib
import pandas as pd
import pickle
import time
from collections import defaultdict

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import seaborn as sns

import plotly.offline as pyo
# Set notebook mode to work in offline
pyo.init_notebook_mode()

import uqef

In [ ]:
sys.path.insert(1, '/work/ga45met/mnt/linux_cluster_2/UQEF-Dynamic')
from uqef_dynamic.utils import utility
from uqef_dynamic.utils import uqPostprocessing
from uqef_dynamic.models.ishigami import IshigamiModel
from uqef_dynamic.models.ishigami import IshigamiStatistics


- [m1] MC (samples-based) with pick-freeze approach
- [m2] Saltelli/MC (samples-based) with rank-based approach
- [m3] gPCE+SC regression approach...
- [m4] gPCE+PSP with a full grid and polynomials of total-order
- [m5] gPCE+PSP with sparse grid and polynomials of total-order
- [m6] gPCE+PSP with a full grid and sparse polynomials (hyperbolic truncation)
- [m7] gPCE+PSP with sparse grid and sparse polynomials (hyperbolic truncation)



# Plot Convergence Plots

In [ ]:
a = 7
b = 0.1
ishigamiModelObject = IshigamiModel.IshigamiModel(configurationObject=None, a=a, b=b)

x1 = cp.Uniform(-math.pi, math.pi)
x2 = cp.Uniform(-math.pi, math.pi)
x3 = cp.Uniform(-math.pi, math.pi)
joint_isghigami = cp.J(x1, x2, x3)
joint_isghigami_standard = cp.J(cp.Uniform(-1,1), cp.Uniform(-1,1), cp.Uniform(-1,1))

In [ ]:
scratch_dir = pathlib.Path("/work/ga45met/ishigami_runs/simulations_sep_2024")
workingDir = scratch_dir / "saltelli_1000_random"
qoi_string="Value"
timestamp=0.0
dict_with_results_of_interest = uqPostprocessing.read_all_saved_uqef_dynamic_results_and_produce_dict_of_interest_single_qoi_single_timestamp(
    workingDir=workingDir, 
    timestamp=timestamp, qoi_column_name=qoi_string,
    plotting=False, model=ishigamiModelObject,
    analytical_E=3.48227783540168,
    analytical_Var=13.887058470972093,
    analytical_Sobol_m=np.array([0.3138, 0.4424, 0.0], dtype=np.float64),
    analytical_Sobol_t=np.array([0.5574, 0.4424, 0.2436], dtype=np.float64),
    compare_surrogate_and_original_model=True
)
dict_with_results_of_interest

In [ ]:
def read_all_data_from_workingDir(workingDir, scratch_dir):
    workingDir = scratch_dir / workingDir
    qoi_string="Value"
    timestamp=0.0
    dict_with_results_of_interest = uqPostprocessing.read_all_saved_uqef_dynamic_results_and_produce_dict_of_interest_single_qoi_single_timestamp(
        workingDir=workingDir, 
        timestamp=timestamp, qoi_column_name=qoi_string,
        plotting=False, model=ishigamiModelObject,
        analytical_E=3.48227783540168,
        analytical_Var=13.887058470972093,
        analytical_Sobol_t=np.array([0.5574, 0.4424, 0.2436], dtype=np.float64),
        analytical_Sobol_m=np.array([0.3138, 0.4424, 0.0], dtype=np.float64),
        compare_surrogate_and_original_model=True
    )
    return dict_with_results_of_interest

# Iterarting Over Multiple

In [ ]:
list_of_runs_for_plotting = [
    "mc_100_random",
    "mc_100_sobol",
    "mc_100_latin_hypercube",
    "mc_100_halton",
    "mc_1000_random",
    "mc_1000_sobol",
    "mc_1000_latin_hypercube",
    "mc_1000_halton",
    "mc_10000_random",
    "mc_10000_sobol",
    "mc_10000_latin_hypercube",
    "mc_10000_halton",
    "mc_100000_random",
    "mc_100000_sobol",
    "mc_100000_latin_hypercube",
    "mc_100000_halton",
    "saltelli_10_random",
    "saltelli_10_sobol",
    "saltelli_10_latin_hypercube",
    "saltelli_10_halton",
    "saltelli_100_random",
    "saltelli_100_sobol",
    "saltelli_100_latin_hypercube",
    "saltelli_100_halton",
    "saltelli_1000_random",
    "saltelli_1000_sobol",
    "saltelli_1000_latin_hypercube",
    "saltelli_1000_halton",
    "saltelli_10000_random",
    "saltelli_10000_sobol",
    "saltelli_10000_latin_hypercube",
    "saltelli_10000_halton",

]

In [ ]:
scratch_dir = pathlib.Path("/work/ga45met/ishigami_runs/simulations_sep_2024")
model_output = scratch_dir

# list all the subdirectories
assert model_output.is_dir()
sub_folder_list = []
for x in model_output.iterdir():
    if x.is_dir():
        sub_folder_list.append(x)

list_of_runs_for_plotting_exist =[]
for single_run in list_of_runs_for_plotting:
    x = model_output / single_run
    if x.is_dir() and x in sub_folder_list:
#         print(x)
        list_of_runs_for_plotting_exist.append(x)
    else:
        print(x)
    
id_run_list = list(range(len(list_of_runs_for_plotting_exist)))

print(f"list_of_runs_for_plotting: {len(list_of_runs_for_plotting)}")
print(f"list_of_runs_for_plotting_exist: {len(list_of_runs_for_plotting_exist)}")

list_dict_runs = []
for id_run in id_run_list:
    single_run_workingDir = list_of_runs_for_plotting_exist[id_run]
    sinlge_run_dict_results = read_all_data_from_workingDir(single_run_workingDir, scratch_dir)
    list_dict_runs.append(sinlge_run_dict_results)
df = pd.DataFrame(list_dict_runs)
stochasticParameterNames = sinlge_run_dict_results["stochasticParameterNames"]

# # Unpack the array column into separate columns
# columns_to_unpact = ["sobol_m", "sobol_m_error", "sobol_t", "sobol_t_error"]
# for single_column_to_unpack in columns_to_unpact:
#     df[single_column_to_unpack] = df[single_column_to_unpack].apply(lambda x: x if isinstance(x, list) else [])
#     new_column_names = [f"{single_column_to_unpack}_{single_param_name}" for single_param_name in stochasticParameterNames]
#     unpacked_df = pd.DataFrame(df[single_column_to_unpack].tolist(), index=df.index, columns=new_column_names)
#     # Combine with the original DataFrame, excluding the original single_column_to_unpack column
#     df = df.drop(single_column_to_unpack, axis=1).join(unpacked_df)
# df

condition1 = (df['variant']=="m1") & (df["sampling_rule"]=="random")
condition2 = (df['variant']=="m1") & (df["sampling_rule"]=="sobol")
condition3 = (df['variant']=="m1") & (df["sampling_rule"]=="latin_hypercube")
condition4 = (df['variant']=="m1") & (df["sampling_rule"]=="halton")
condition5 = (df['variant']=="m2") & (df["sampling_rule"]=="random")
condition6 = (df['variant']=="m2") & (df["sampling_rule"]=="sobol")
condition7 = (df['variant']=="m2") & (df["sampling_rule"]=="latin_hypercube")
condition8 = (df['variant']=="m2") & (df["sampling_rule"]=="halton")
df.loc[condition1, "condition"]="mc-r"
df.loc[condition2, "condition"]="mc-s"
df.loc[condition3, "condition"]="mc-lh"
df.loc[condition4, "condition"]="mc-h"
df.loc[condition5, "condition"]="sal-r"
df.loc[condition6, "condition"]="sal-s"
df.loc[condition7, "condition"]="sal-lh"
df.loc[condition8, "condition"]="sal-h"

In [ ]:
df

In [ ]:
df.columns

## Plotting

In [ ]:
log_x=True
log_y=False
mode_name  = "ishigami"
y="error_mean" # "error_model_l2_scaled" "error_model_linf" "error_model_l2" "error_mean" "error_var" "sobol_m_error_x1" "sobol_t_error_x1"

#df = df.sort_values(by=y)
    
# px.line vs px.scatter
fig = px.scatter(
    df, x="number_full_model_evaluations", y=y, 
    title=f"Convergence plot {mode_name} - {y} (MC and Saltellis)",
    text="sampling_rule",
    color="condition", 
    symbol="condition", 
    hover_name="number_full_model_evaluations", 
    log_x=log_x, log_y=log_y,
    range_x=[df["number_full_model_evaluations"].min()-5,df["number_full_model_evaluations"].max()+5]
)
fig.update_traces(mode="markers+lines")
# fig.update_traces(mode="markers")
# fig.update_layout(hovermode="x unified")
fig.update_layout(
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    legend_title="Method",
    xaxis_title="Number Full Model Evaluations",
    yaxis_title=y,
)
fig.show()

In [ ]:
log_x=True
log_y=False
mode_name  = "ishigami"
y="error_var" # "error_model_l2_scaled" "error_model_linf" "error_model_l2" "error_mean" "error_var" "sobol_m_error_x1" "sobol_t_error_x1"

#df = df.sort_values(by=y)
    
# px.line vs px.scatter
fig = px.scatter(
    df, x="number_full_model_evaluations", y=y, 
    title=f"Convergence plot {mode_name} - {y} (MC and Saltellis)",
    text="sampling_rule",
    color="condition", 
    symbol="condition", 
    hover_name="number_full_model_evaluations", 
    log_x=log_x, log_y=log_y,
    range_x=[df["number_full_model_evaluations"].min()-5,df["number_full_model_evaluations"].max()+5]
)
fig.update_traces(mode="markers+lines")
# fig.update_traces(mode="markers")
# fig.update_layout(hovermode="x unified")
fig.update_layout(
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    legend_title="Method",
    xaxis_title="Number Full Model Evaluations",
    yaxis_title=y,
)
fig.show()

In [ ]:
log_x=True
log_y=False
mode_name  = "ishigami"
y="sobol_m_x1" # "error_model_l2_scaled" "error_model_linf" "error_model_l2" "error_mean" "error_var" "sobol_m_error_x1" "sobol_t_error_x1"

#df = df.sort_values(by=y)
    
# px.line vs px.scatter
fig = px.scatter(
    df, x="number_full_model_evaluations", y=y, 
    title=f"Convergence plot {mode_name} - {y} (MC and Saltellis)",
    text="sampling_rule",
    color="condition", 
    symbol="condition", 
    hover_name="number_full_model_evaluations", 
    log_x=log_x, log_y=log_y,
    range_x=[df["number_full_model_evaluations"].min()-5,df["number_full_model_evaluations"].max()+5]
)
fig.update_traces(mode="markers+lines")
# fig.update_traces(mode="markers")
# fig.update_layout(hovermode="x unified")
fig.update_layout(
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    legend_title="Method",
    xaxis_title="Number Full Model Evaluations",
    yaxis_title=y,
)
fig.show()

In [ ]:
log_x=True
log_y=False
mode_name  = "ishigami"
y="sobol_m_x1_error" # "error_model_l2_scaled" "error_model_linf" "error_model_l2" "error_mean" "error_var" "sobol_m_error_x1" "sobol_t_error_x1"

# df = df.sort_values(by=y)
    
# px.line vs px.scatter
fig = px.scatter(
    df, x="number_full_model_evaluations", y=y, 
    title=f"Convergence plot {mode_name} - {y} (MC and Saltellis)",
    text="sampling_rule",
    color="condition", 
    symbol="condition", 
    hover_name="number_full_model_evaluations", 
    log_x=log_x, log_y=log_y,
    range_x=[df["number_full_model_evaluations"].min()-5,df["number_full_model_evaluations"].max()+5]
)
fig.update_traces(mode="markers+lines")
# fig.update_traces(mode="markers")
# fig.update_layout(hovermode="x unified")
fig.update_layout(
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    legend_title="Method",
    xaxis_title="Number Full Model Evaluations",
    yaxis_title=y,
)
fig.show()

In [ ]:
log_x=True
log_y=True
mode_name  = "ishigami"
y="sobol_m_x2_error" # "error_model_l2_scaled" "error_model_linf" "error_model_l2" "error_mean" "error_var" "sobol_m_error_x1" "sobol_t_error_x1"

# df = df.sort_values(by=y)
    
# px.line vs px.scatter
fig = px.scatter(
    df, x="number_full_model_evaluations", y=y, 
    title=f"Convergence plot {mode_name} - {y} (MC and Saltellis)",
    text="sampling_rule",
    color="condition", 
    symbol="condition", 
    hover_name="number_full_model_evaluations", 
    log_x=log_x, log_y=log_y,
    range_x=[df["number_full_model_evaluations"].min()-5,df["number_full_model_evaluations"].max()+5]
)
fig.update_traces(mode="markers+lines")
# fig.update_traces(mode="markers")
# fig.update_layout(hovermode="x unified")
fig.update_layout(
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    legend_title="Method",
    xaxis_title="Number Full Model Evaluations",
    yaxis_title=y,
)
fig.show()

In [ ]:
log_x=True
log_y=False
mode_name  = "ishigami"
y="sobol_m_x3_error" # "error_model_l2_scaled" "error_model_linf" "error_model_l2" "error_mean" "error_var" "sobol_m_error_x1" "sobol_t_error_x1"

# df = df.sort_values(by=y)
    
# px.line vs px.scatter
fig = px.scatter(
    df, x="number_full_model_evaluations", y=y, 
    title=f"Convergence plot {mode_name} - {y} (MC and Saltellis)",
    text="sampling_rule",
    color="condition", 
    symbol="condition", 
    hover_name="number_full_model_evaluations", 
    log_x=log_x, log_y=log_y,
    range_x=[df["number_full_model_evaluations"].min()-5,df["number_full_model_evaluations"].max()+5]
)
fig.update_traces(mode="markers+lines")
# fig.update_traces(mode="markers")
# fig.update_layout(hovermode="x unified")
fig.update_layout(
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    legend_title="Method",
    xaxis_title="Number Full Model Evaluations",
    yaxis_title=y,
)
fig.show()